# GLIDE vs FLEXPART footprint comparison

Side-by-side comparison of a GLIDE surface footprint against the FLEXPART reference fixture in `data/FLEXPART/`. Both are reduced to STILT-style surface sensitivity (`(mol/mol)/(mol/m²/s)`) on the **same lat/lon grid** (the GLIDE run is configured cell-for-cell against the fixture), so the panels are directly comparable.

GLIDE stores a raw residence-time footprint (`s` per cell); `lpdm.comparison.to_stilt_surface_footprint` converts it to the same units FLEXPART reports in `srr`. The surface layer matches on both sides (FLEXPART `LPDM_sampling_height = 40 m`, GLIDE bottom z-bin `0–40 m`), so the conversion is exact — no partial-overlap approximation.

This notebook is the comparison companion to `footprint_explorer.ipynb`; start here once a GLIDE run's `release_time`s overlap the fixture's.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr

import holoviews as hv
import geoviews as gv
import hvplot.xarray  # noqa: F401  registers the .hvplot accessor on DataArray

from lpdm.comparison import to_stilt_surface_footprint

hv.extension("bokeh")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
GLIDE_ZARR = PROJECT_ROOT / "outputs" / "mhd-202401-hourly" / "footprints.zarr"
FLEXPART_NC = PROJECT_ROOT / "data" / "FLEXPART" / "FLEXPART_MHD_test_202401.nc"

# Surface-layer depth for the STILT conversion. Must match FLEXPART's
# LPDM_sampling_height and the GLIDE run's bottom z-bin edge (both 40 m here)
# for an exact (non-approximated) conversion.
SURFACE_LAYER_DEPTH_M = 40.0

print(f"GLIDE store:   {GLIDE_ZARR}  (exists: {GLIDE_ZARR.exists()})")
print(f"FLEXPART file: {FLEXPART_NC}  (exists: {FLEXPART_NC.exists()})")

## Load both datasets and find overlapping release times

The two products are matched on `release_time`. The FLEXPART fixture carries a subset of January 2024 (days 1, 10, 20, 30 × 24 h); only the GLIDE releases whose timestamps appear in both can be compared.

In [ ]:
glide_ds = xr.open_zarr(GLIDE_ZARR).load()
glide_fp = glide_ds["footprint"]

flex_ds = xr.open_dataset(FLEXPART_NC, engine="h5netcdf")
flex_srr = flex_ds["srr"]  # (time, latitude, longitude), units (mol/mol)/(mol/m^2/s)

release_lon = float(glide_ds.attrs["release_lon"])
release_lat = float(glide_ds.attrs["release_lat"])

common_times = np.intersect1d(glide_fp["release_time"].values, flex_srr["time"].values)
if common_times.size == 0:
    raise ValueError(
        "No overlapping release times between the GLIDE run and the FLEXPART fixture. "
        "Check that the GLIDE run covers one of the fixture days (1, 10, 20, 30 Jan 2024)."
    )

# Grids must match cell-for-cell for a direct comparison.
assert np.allclose(glide_fp["latitude"].values, flex_srr["latitude"].values), "latitude grids differ"
assert np.allclose(glide_fp["longitude"].values, flex_srr["longitude"].values), "longitude grids differ"

print(f"GLIDE releases:    {glide_fp.sizes['release_time']}")
print(f"FLEXPART releases: {flex_srr.sizes['time']}")
print(f"overlapping:       {common_times.size}")
print(f"first overlapping: {np.datetime_as_string(common_times[0], unit='h')}")
print(f"last overlapping:  {np.datetime_as_string(common_times[-1], unit='h')}")
print(f"release point:     ({release_lon:.4f}, {release_lat:.4f})")

## Select a release and convert GLIDE to STILT units

Change `RELEASE_INDEX` to compare a different overlapping release. The GLIDE→STILT conversion uses the near-surface air density FLEXPART reported for that release (derived from its `air_pressure`/`air_temperature`), so the two magnitudes are on a faithful common footing rather than a standard-atmosphere approximation.

In [ ]:
RELEASE_INDEX = 0  # index into common_times
release_time = common_times[RELEASE_INDEX]

# Near-surface air density from the FLEXPART-reported p, T at this release
# (air_pressure is hPa, air_temperature is K). rho = p / (R_d T).
R_DRY = 287.05
flex_at_release = flex_ds.sel(time=release_time)
p_pa = float(flex_at_release["air_pressure"]) * 100.0
T_k = float(flex_at_release["air_temperature"])
air_density = p_pa / (R_DRY * T_k)

glide_raw = glide_fp.sel(release_time=release_time)  # (time_ago, z_bin, lat, lon)
glide_stilt = to_stilt_surface_footprint(
    glide_raw,
    surface_layer_depth_m=SURFACE_LAYER_DEPTH_M,
    air_density_kg_m3=air_density,
    integrate_time=True,
)  # (lat, lon), units m^2 s mol^-1  ==  (mol/mol)/(mol/m^2/s)
flex_stilt = flex_srr.sel(time=release_time)  # (lat, lon), same units

print(f"release_time: {np.datetime_as_string(release_time, unit='h')}   air_density={air_density:.4f} kg/m^3")
print(f"GLIDE    total={float(glide_stilt.sum()):.4e}  max={float(glide_stilt.max()):.4e}  nonzero cells={int((glide_stilt > 0).sum())}")
print(f"FLEXPART total={float(flex_stilt.sum()):.4e}  max={float(flex_stilt.max()):.4e}  nonzero cells={int((flex_stilt > 0).sum())}")

## Side-by-side maps

Both panels share a log colour scale (computed from the union of the two fields' positive values) so colours mean the same sensitivity on each side. The cyan dot is the Mace Head release point. Pan/zoom are linked across the two maps.

In [ ]:
# Shared colour limits from the union of both positive fields.
glide_pos = glide_stilt.values[glide_stilt.values > 0]
flex_pos = flex_stilt.values[flex_stilt.values > 0]
both_pos = np.concatenate([glide_pos, flex_pos])
clim = (float(both_pos.min()), float(both_pos.max()))

release_marker = gv.Points([(release_lon, release_lat)]).opts(
    color="cyan", size=9, line_color="black",
)


def surface_map(field: xr.DataArray, title: str):
    """Log-scaled STILT surface-sensitivity raster on a tile basemap."""
    positive = field.where(field > 0)  # mask zeros so the log scale + transparency behave
    raster = positive.hvplot.image(
        x="longitude",
        y="latitude",
        tiles="CartoLight",
        cmap="magma",
        logz=True,
        clim=clim,
        geo=True,
        project=False,
        height=500,
        width=600,
        alpha=0.85,
        colorbar=True,
        clabel="surface sensitivity (mol/mol)/(mol/m\u00b2/s)",
        title=title,
    )
    return raster * release_marker


stamp = np.datetime_as_string(release_time, unit="h")
layout = (
    surface_map(glide_stilt, f"GLIDE  {stamp}")
    + surface_map(flex_stilt, f"FLEXPART  {stamp}")
).cols(2)
layout

## CH₄ timeseries comparison

Multiply each footprint by the EDGAR v2025 annual-mean CH₄ flux (regridded to this grid, `mol m⁻² s⁻¹`) and sum over the domain to get a modelled concentration enhancement at Mace Head. Both GLIDE and FLEXPART footprints are converted to STILT units first, so the scalar product has units of mol/mol.

$$\Delta c(t) = \sum_{i,j} f_{ij}(t) \cdot q_{ij}$$

where $f_{ij}$ is in (mol/mol)/(mol m⁻² s⁻¹) and $q_{ij}$ is the EDGAR flux in mol m⁻² s⁻¹.

EDGAR 2024 is a single annual-mean field (no diurnal/seasonal variation), so differences between the two timeseries reflect footprint differences only — the EDGAR flux acts as a fixed spatial weight.

In [ ]:
EDGAR_NC = PROJECT_ROOT / "data" / "EDGAR_CH4_2024_MHD_grid.nc"

edgar_flux = xr.open_dataset(EDGAR_NC)["flux"]  # (lat, lon), mol m-2 s-1

# GLIDE, FLEXPART and EDGAR are all built cell-for-cell on the same nominal grid,
# but their coordinate arrays were constructed independently and differ by float
# rounding (~1e-14). xarray aligns on *exactly* equal coordinate labels, so a bare
# `footprint * flux` inner-joins to the handful of bit-identical cells and silently
# NaNs (→ zeros) the rest. Snap EDGAR and FLEXPART onto the GLIDE footprint's grid
# (verified equal to ~1e-14 by the np.allclose asserts above) so every product
# aligns cell-for-cell.
ref_lat = glide_fp["latitude"].values
ref_lon = glide_fp["longitude"].values
edgar_flux = edgar_flux.assign_coords(latitude=ref_lat, longitude=ref_lon)
flex_srr_aligned = flex_srr.assign_coords(latitude=ref_lat, longitude=ref_lon)

glide_ppb = []
flex_ppb = []
timestamps = []

for t in common_times:
    # Per-release air density from FLEXPART-reported p, T (consistent with map cells above)
    flex_at_t = flex_ds.sel(time=t)
    p_pa = float(flex_at_t["air_pressure"]) * 100.0
    T_k = float(flex_at_t["air_temperature"])
    rho = p_pa / (R_DRY * T_k)

    # GLIDE: raw → STILT units → dot-product with EDGAR flux (×1e9 for mol/mol → ppb)
    glide_raw_t = glide_fp.sel(release_time=t)
    glide_stilt_t = to_stilt_surface_footprint(
        glide_raw_t,
        surface_layer_depth_m=SURFACE_LAYER_DEPTH_M,
        air_density_kg_m3=rho,
        integrate_time=True,
    )
    glide_ppb.append(float((glide_stilt_t * edgar_flux).sum()) * 1e9)

    # FLEXPART: srr already in (mol/mol)/(mol/m²/s)
    flex_ppb.append(float((flex_srr_aligned.sel(time=t) * edgar_flux).sum()) * 1e9)

    timestamps.append(t)

ts = xr.Dataset(
    {
        "GLIDE":     xr.DataArray(glide_ppb, dims="time", attrs={"units": "ppb"}),
        "FLEXPART":  xr.DataArray(flex_ppb,  dims="time", attrs={"units": "ppb"}),
    },
    coords={"time": np.array(timestamps, dtype="datetime64[ns]")},
)

print(ts.to_dataframe().to_string())

In [ ]:
glide_line = ts["GLIDE"].hvplot.line(
    x="time", label="GLIDE", color="#e6550d", line_width=2,
)
flex_line = ts["FLEXPART"].hvplot.line(
    x="time", label="FLEXPART", color="#3182bd", line_width=2, line_dash="dashed",
)

(glide_line * flex_line).opts(
    xlabel="Release time (UTC)",
    ylabel="ΔCH₄ (ppb)",
    title="Modelled CH₄ enhancement at Mace Head — EDGAR 2024 flux × footprint",
    legend_position="top_right",
    height=400,
    width=900,
    toolbar="above",
)